# Phase 2: Feature Extraction
**Speech Emotion Recognition — CREMA-D Dataset**

Extracts two types of features from all WAV files:
- **Pipeline A — MFCC**: flat 80-dim vectors (for FC-NN) and sequences of shape `(200, 40)` (for LSTM & Transformer)
- **Pipeline B — Mel-Spectrogram**: saved as `(224, 224, 3)` PNG images (for CNN & ResNet-18)

**Run `data_preparation.ipynb` first** to generate `metadata.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving images
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

print('Libraries loaded.')

## 1. Configuration

In [ ]:
BASE_DIR      = os.path.dirname(os.path.abspath('__file__'))
METADATA_CSV  = os.path.join(BASE_DIR, 'metadata.csv')
FEATURES_DIR  = os.path.join(BASE_DIR, 'features')
SPECTROGRAM_DIR = os.path.join(BASE_DIR, 'spectrograms')

os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(SPECTROGRAM_DIR, exist_ok=True)

# Audio settings
SAMPLE_RATE   = 22050
DURATION      = 3.0          # seconds — clips longer are trimmed, shorter are padded
N_MFCC        = 40           # number of MFCC coefficients
N_MELS        = 128          # mel filterbanks for spectrogram
MAX_FRAMES    = 200          # fixed sequence length for LSTM/Transformer
IMG_SIZE      = (224, 224)   # ResNet-18 compatible size

print(f'Output directories created:')
print(f'  Features     : {FEATURES_DIR}')
print(f'  Spectrograms : {SPECTROGRAM_DIR}')

In [ ]:
df = pd.read_csv(METADATA_CSV)
print(f'Loaded metadata: {len(df)} records')
df['split'].value_counts()

## 2. Helper Functions

In [ ]:
def load_audio(filepath, sr=SAMPLE_RATE, duration=DURATION):
    """Load audio, pad/trim to fixed duration."""
    y, _ = librosa.load(filepath, sr=sr, duration=duration)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y


def extract_mfcc_flat(y, sr=SAMPLE_RATE, n_mfcc=N_MFCC):
    """Extract mean + std of each MFCC → 80-dim vector."""
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])


def extract_mfcc_seq(y, sr=SAMPLE_RATE, n_mfcc=N_MFCC, max_frames=MAX_FRAMES):
    """Extract MFCC matrix, pad/trim to (max_frames, n_mfcc)."""
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T  # (T, 40)
    if mfcc.shape[0] < max_frames:
        pad = np.zeros((max_frames - mfcc.shape[0], n_mfcc))
        mfcc = np.vstack([mfcc, pad])
    else:
        mfcc = mfcc[:max_frames]
    return mfcc  # (200, 40)


def save_melspectrogram(y, sr, save_path, img_size=IMG_SIZE):
    """Compute Mel-spectrogram and save as (224, 224, 3) PNG."""
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    librosa.display.specshow(mel_db, sr=sr, x_axis=None, y_axis=None, ax=ax)
    ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(save_path, dpi=100)
    plt.close(fig)

    # Ensure exactly (224, 224, 3)
    img = Image.open(save_path).convert('RGB').resize(img_size, Image.LANCZOS)
    img.save(save_path)


print('Helper functions defined.')

## 3. Preview — Sample Audio Visualization

In [ ]:
matplotlib.use('inline' if 'inline' in plt.get_backend() else 'Agg')
%matplotlib inline

sample_row = df[df['emotion_code'] == 'ANG'].iloc[0]
y_sample, sr = librosa.load(sample_row['filepath'], sr=SAMPLE_RATE)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Waveform
librosa.display.waveshow(y_sample, sr=sr, ax=axes[0])
axes[0].set_title(f'Waveform — {sample_row["emotion_name"]}')

# MFCC
mfcc = librosa.feature.mfcc(y=y_sample, sr=sr, n_mfcc=40)
img = librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=axes[1])
axes[1].set_title('MFCC (40 coefficients)')
fig.colorbar(img, ax=axes[1])

# Mel-Spectrogram
mel = librosa.feature.melspectrogram(y=y_sample, sr=sr, n_mels=128)
mel_db = librosa.power_to_db(mel, ref=np.max)
img2 = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[2])
axes[2].set_title('Mel-Spectrogram (dB)')
fig.colorbar(img2, ax=axes[2], format='%+2.0f dB')

plt.tight_layout()
plt.show()

## 4. Pipeline A — MFCC Extraction (Flat + Sequential)

In [ ]:
for split in ['train', 'val', 'test']:
    split_df = df[df['split'] == split].reset_index(drop=True)
    n = len(split_df)

    mfcc_flat = np.zeros((n, N_MFCC * 2), dtype=np.float32)   # (n, 80)
    mfcc_seq  = np.zeros((n, MAX_FRAMES, N_MFCC), dtype=np.float32)  # (n, 200, 40)
    labels    = split_df['emotion_label'].values

    errors = 0
    for i, row in tqdm(split_df.iterrows(), total=n, desc=f'MFCC [{split}]'):
        try:
            y = load_audio(row['filepath'])
            mfcc_flat[i] = extract_mfcc_flat(y)
            mfcc_seq[i]  = extract_mfcc_seq(y)
        except Exception as e:
            errors += 1
            print(f'Error on {row["filename"]}: {e}')

    np.save(os.path.join(FEATURES_DIR, f'mfcc_flat_{split}.npy'), mfcc_flat)
    np.save(os.path.join(FEATURES_DIR, f'mfcc_seq_{split}.npy'),  mfcc_seq)
    np.save(os.path.join(FEATURES_DIR, f'labels_{split}.npy'),    labels)

    print(f'[{split}] flat: {mfcc_flat.shape}, seq: {mfcc_seq.shape}, errors: {errors}')

print('MFCC extraction complete.')

## 5. Pipeline B — Mel-Spectrogram Images

In [ ]:
EMOTION_CODES = ['ANG', 'DIS', 'FEA', 'HAP', 'NEU', 'SAD']

# Create folder structure: spectrograms/{split}/{emotion}/
for split in ['train', 'val', 'test']:
    for emotion in EMOTION_CODES:
        os.makedirs(os.path.join(SPECTROGRAM_DIR, split, emotion), exist_ok=True)

print('Folder structure created.')

In [ ]:
errors = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc='Mel-Spectrograms'):
    out_path = os.path.join(
        SPECTROGRAM_DIR,
        row['split'],
        row['emotion_code'],
        row['filename'].replace('.wav', '.png')
    )
    if os.path.exists(out_path):
        continue  # skip if already extracted (resume-safe)
    try:
        y = load_audio(row['filepath'])
        save_melspectrogram(y, SAMPLE_RATE, out_path)
    except Exception as e:
        errors += 1
        print(f'Error on {row["filename"]}: {e}')

print(f'Mel-spectrogram extraction complete. Errors: {errors}')

## 6. Verify Outputs

In [ ]:
print('=== MFCC Feature Files ===')
for split in ['train', 'val', 'test']:
    flat = np.load(os.path.join(FEATURES_DIR, f'mfcc_flat_{split}.npy'))
    seq  = np.load(os.path.join(FEATURES_DIR, f'mfcc_seq_{split}.npy'))
    lbl  = np.load(os.path.join(FEATURES_DIR, f'labels_{split}.npy'))
    print(f'  [{split}] flat={flat.shape}, seq={seq.shape}, labels={lbl.shape}')

print()
print('=== Spectrogram Image Counts ===')
for split in ['train', 'val', 'test']:
    total = sum(
        len(os.listdir(os.path.join(SPECTROGRAM_DIR, split, em)))
        for em in EMOTION_CODES
        if os.path.exists(os.path.join(SPECTROGRAM_DIR, split, em))
    )
    print(f'  [{split}] {total} images')

In [ ]:
# Show sample spectrogram images for each emotion
%matplotlib inline
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, emotion in enumerate(EMOTION_CODES):
    folder = os.path.join(SPECTROGRAM_DIR, 'train', emotion)
    files = os.listdir(folder)
    img = Image.open(os.path.join(folder, files[0]))
    axes[idx].imshow(img)
    axes[idx].set_title(f'{emotion}', fontsize=13, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Sample Mel-Spectrograms per Emotion', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| Feature | Shape | Used by |
|---------|-------|---------|
| `mfcc_flat_{split}.npy` | `(N, 80)` | FC-NN |
| `mfcc_seq_{split}.npy` | `(N, 200, 40)` | LSTM, Transformer |
| `spectrograms/{split}/{emotion}/*.png` | `(224, 224, 3)` | CNN, ResNet-18 |

**Next step:** Run model notebooks — start with `model_fcnn.ipynb`.